In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import os

In [2]:
list_files = os.listdir("../")
list_files

['away_team.csv',
 'away_team_score.csv',
 'event.csv',
 'home_team.csv',
 'home_team_score.csv',
 'Javadi',
 'notebook.ipynb',
 'odds.csv',
 'pbp.csv',
 'power.csv',
 'round.csv',
 'season.csv',
 'statistics.csv',
 'Tennis-project.zip',
 'time.csv',
 'tournament.csv',
 'venue.csv',
 'votes.csv']

In [3]:
event_df = pd.read_csv("../event.csv")
home_team_df = pd.read_csv("../home_team.csv")
away_team_df = pd.read_csv("../away_team.csv")

In [4]:


# ─────────────────────────────────────────────────────────────────────────────
# THRESHOLDS — based on real tennis records
# ─────────────────────────────────────────────────────────────────────────────
HEIGHT_MIN_MEN   = 160    # cm — below Olivier Rochus (168cm, shortest modern ATP)
HEIGHT_MAX_MEN   = 215    # cm — above Karlovic/Opelka (211cm, tallest ever ATP)
HEIGHT_MIN_WOMEN = 155    # cm — below shortest WTA players on record
HEIGHT_MAX_WOMEN = 200    # cm — safely above tallest WTA players (~186cm)
RANK_MIN         = 1
RANK_MAX         = 2000   # active ranked players on ATP/WTA

# Units detection thresholds
# If height values cluster around 1.5–2.2, they're in metres → convert to cm
METRES_MAX       = 3.0    # anything below this is likely metres not cm

report         = []
total_original_home = len(home_team_df)
total_original_away = len(away_team_df)
total_original      = total_original_home + total_original_away

def log(step, desc, removed, note=''):
    report.append({
        'Step'          : step,
        'Description'   : desc,
        'Rows Removed'  : removed,
        '% of Original' : round(removed / total_original * 100, 2),
        'Note'          : note
    })

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Combine home and away player tables
# Both tables have the same player columns — a player can appear
# as home in some matches and away in others, so we need both
# ─────────────────────────────────────────────────────────────────────────────
cols_needed = [
    'match_id', 'player_id', 'name', 'full_name',
    'height', 'current_rank', 'gender', 'country'
]

home_players = home_team_df[cols_needed].copy()
away_players = away_team_df[cols_needed].copy()

# Tag which table each row came from (useful for debugging)
home_players['source'] = 'home'
away_players['source'] = 'away'

# Combine into one big table
all_players = pd.concat([home_players, away_players], ignore_index=True)
print(f"Combined table size: {len(all_players):,} rows")

# ── Step 1: Print the raw rank distribution BEFORE any cutoff ────────────────
print("Rank distribution (raw):")
print(all_players['current_rank'].describe(
    percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
))

# ── Step 2: Only remove clearly impossible values ────────────────────────────
# Rank <= 0 is always wrong (no such thing as rank 0 or negative rank)
# Rank > 5000 is almost certainly a data entry error or placeholder
# (the entire professional + challenger + ITF system combined 
#  has fewer than 5000 ranked players at any given time)
bad_rank = (
    all_players['current_rank'].notna() &
    (
        (all_players['current_rank'] <= 0) |
        (all_players['current_rank'] > 5000)
    )
)
rank_nulled = bad_rank.sum()
all_players.loc[bad_rank, 'current_rank'] = np.nan

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Drop rows with null player_id — can't identify the player
# ─────────────────────────────────────────────────────────────────────────────
before       = len(all_players)
all_players  = all_players[all_players['player_id'].notna()].copy()
removed      = before - len(all_players)
log(2, 'Null player_id rows dropped', removed, '')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Detect and fix height units
# Check if heights look like metres (1.5–2.2) or centimetres (155–215)
# ─────────────────────────────────────────────────────────────────────────────
non_null_heights = all_players['height'].dropna()
median_height    = non_null_heights.median()

print(f"\n  Height column — median value: {median_height}")
print(f"  Height column — min: {non_null_heights.min()}, max: {non_null_heights.max()}")

if median_height < METRES_MAX:
    # Heights are in metres — convert to centimetres
    print("  → Heights appear to be in METRES. Converting to centimetres.")
    all_players['height'] = all_players['height'] * 100
    log(3, 'Heights converted from metres to centimetres', 0,
        f'Detected metres: median was {median_height:.2f}m')
else:
    print("  → Heights appear to be in CENTIMETRES. No conversion needed.")
    log(3, 'Height unit check — no conversion needed', 0,
        f'Median height: {median_height:.1f}cm')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Standardise gender labels
# ─────────────────────────────────────────────────────────────────────────────
before = len(all_players)
all_players['gender'] = all_players['gender'].str.strip().str.upper()
all_players['gender'] = all_players['gender'].map(
    lambda x: 'M' if x in ['M', 'MALE', 'MEN', '1', 'ATP']
    else ('F' if x in ['F', 'FEMALE', 'WOMEN', '2', 'WTA']
    else np.nan)
)
all_players = all_players[all_players['gender'].notna()].copy()
removed     = before - len(all_players)
log(4, 'Rows with null or unrecognised gender labels', removed, '')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Null out invalid height values per gender
# We don't drop rows yet — a player might still be usable for rank
# analysis even if their height is bad (though they'll be excluded later)
# ─────────────────────────────────────────────────────────────────────────────
cells_nulled = 0

# Men
men_mask  = all_players['gender'] == 'M'
bad_height_men = (
    men_mask &
    all_players['height'].notna() &
    (
        (all_players['height'] == 0) |
        (all_players['height'] < HEIGHT_MIN_MEN) |
        (all_players['height'] > HEIGHT_MAX_MEN)
    )
)
cells_nulled += bad_height_men.sum()
all_players.loc[bad_height_men, 'height'] = np.nan

# Women
women_mask = all_players['gender'] == 'F'
bad_height_women = (
    women_mask &
    all_players['height'].notna() &
    (
        (all_players['height'] == 0) |
        (all_players['height'] < HEIGHT_MIN_WOMEN) |
        (all_players['height'] > HEIGHT_MAX_WOMEN)
    )
)
cells_nulled += bad_height_women.sum()
all_players.loc[bad_height_women, 'height'] = np.nan

report.append({
    'Step'          : 5,
    'Description'   : 'Invalid height values nulled (out of realistic range)',
    'Rows Removed'  : f'{cells_nulled} cells nulled',
    '% of Original' : round(cells_nulled / total_original * 100, 2),
    'Note'          : f'Men: {HEIGHT_MIN_MEN}–{HEIGHT_MAX_MEN}cm | '
                      f'Women: {HEIGHT_MIN_WOMEN}–{HEIGHT_MAX_WOMEN}cm'
})

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: Null out invalid rank values
# ─────────────────────────────────────────────────────────────────────────────
bad_rank = (
    all_players['current_rank'].notna() &
    (
        (all_players['current_rank'] <= 0) |
        (all_players['current_rank'] > RANK_MAX)
    )
)
rank_nulled = bad_rank.sum()
all_players.loc[bad_rank, 'current_rank'] = np.nan

report.append({
    'Step'          : 6,
    'Description'   : 'Invalid rank values nulled (<=0 or >2000)',
    'Rows Removed'  : f'{rank_nulled} cells nulled',
    '% of Original' : round(rank_nulled / total_original * 100, 2),
    'Note'          : ''
})

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7: Deduplicate — one row per player
# Strategy:
#   height    → median across all appearances (height never changes)
#   rank      → most recent rank (rank DOES change — use latest match)
#   name      → most frequent name (handles minor spelling differences)
#   country   → most frequent country
# ─────────────────────────────────────────────────────────────────────────────

# First merge with event_df to get match dates for rank recency
date_lookup = event_df[['match_id', 'start_datetime']].copy()
date_lookup = date_lookup[date_lookup['start_datetime'].notna()]
date_lookup = date_lookup[date_lookup['start_datetime'] > 0]
date_lookup['match_date'] = pd.to_datetime(
    date_lookup['start_datetime'], unit='s', utc=True
).dt.tz_localize(None)

all_players = all_players.merge(
    date_lookup[['match_id', 'match_date']],
    on='match_id', how='left'
)

# Sort by match date so the most recent row is last per player
all_players = all_players.sort_values('match_date', na_position='first')

# For each player, aggregate across all their match appearances
player_df = (
    all_players
    .groupby(['player_id', 'gender'])
    .agg(
        # Most frequent name
        name         = ('full_name', lambda x:
                        x.dropna().value_counts().index[0]
                        if x.dropna().any() else np.nan),
        # Most frequent country
        country      = ('country', lambda x:
                        x.dropna().value_counts().index[0]
                        if x.dropna().any() else np.nan),
        # Median height — robust to occasional bad recordings
        height       = ('height', 'median'),
        # Most recent rank — last non-null value after sorting by date
        current_rank = ('current_rank', lambda x:
                        x.dropna().iloc[-1]
                        if x.dropna().any() else np.nan),
        # How many matches they appeared in (useful context)
        match_count  = ('match_id', 'nunique')
    )
    .reset_index()
)

print(f"\n  Unique players after deduplication: {len(player_df):,}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 8: Drop players missing height OR rank
# We need both to compute the correlation
# ─────────────────────────────────────────────────────────────────────────────
before     = len(player_df)
player_df  = player_df[
    player_df['height'].notna() &
    player_df['current_rank'].notna()
].copy()
removed    = before - len(player_df)
log(8, 'Players dropped for missing height or rank', removed,
    'Both values required for correlation analysis')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 9: Check for height inconsistency per player
# After taking the median, flag players where height varied a lot
# across matches (std > 5cm suggests multiple bad recordings)
# ─────────────────────────────────────────────────────────────────────────────
height_std = (
    all_players
    .groupby('player_id')['height']
    .std()
    .reset_index()
    .rename(columns={'height': 'height_std'})
)
player_df = player_df.merge(height_std, on='player_id', how='left')

suspicious_height = player_df['height_std'] > 5
n_suspicious      = suspicious_height.sum()

report.append({
    'Step'          : 9,
    'Description'   : 'Players with height variation >5cm across matches (flagged)',
    'Rows Removed'  : f'{n_suspicious} flagged (kept, using median)',
    '% of Original' : '-',
    'Note'          : 'Median used to reduce impact of bad recordings'
})

# ─────────────────────────────────────────────────────────────────────────────
# STEP 10: Split by gender for separate correlation analyses
# ─────────────────────────────────────────────────────────────────────────────
men_df   = player_df[player_df['gender'] == 'M'].copy()
women_df = player_df[player_df['gender'] == 'F'].copy()

print(f"\n  ATP men with valid height + rank   : {len(men_df):,}")
print(f"  WTA women with valid height + rank : {len(women_df):,}")

# ─────────────────────────────────────────────────────────────────────────────
# CORRELATION ANALYSIS
# Using three methods:
# 1. Pearson  — linear correlation (sensitive to outliers)
# 2. Spearman — rank-based, robust to outliers and non-linearity
# 3. We use Spearman as the primary result since rank data is ordinal
# Note: lower rank number = BETTER player, so we expect a
# NEGATIVE correlation if taller players rank better
# ─────────────────────────────────────────────────────────────────────────────
def compute_correlations(df, label):
    pearson_r,  pearson_p  = stats.pearsonr(
        df['height'], df['current_rank']
    )
    spearman_r, spearman_p = stats.spearmanr(
        df['height'], df['current_rank']
    )

    # Linear regression for the scatter plot trend line
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        df['height'], df['current_rank']
    )

    print(f"\n{'─' * 55}")
    print(f"  CORRELATION RESULTS — {label}")
    print(f"{'─' * 55}")
    print(f"  Sample size  : {len(df):,} players")
    print(f"  Height range : {df['height'].min():.0f}cm – {df['height'].max():.0f}cm")
    print(f"  Rank range   : {int(df['current_rank'].min())} – {int(df['current_rank'].max())}")
    print(f"\n  Pearson r    : {pearson_r:.4f}  (p = {pearson_p:.4f})")
    print(f"  Spearman r   : {spearman_r:.4f}  (p = {spearman_p:.4f})")
    print(f"\n  Primary result (Spearman):")
    print(f"    Correlation : {spearman_r:.4f}")
    significant = spearman_p < 0.05
    direction   = ('Taller players tend to have BETTER rankings'
                   if spearman_r < 0
                   else 'Taller players tend to have WORSE rankings')
    print(f"    Significant : {'YES ✓' if significant else 'NO ✗'} (p = {spearman_p:.4f})")
    if significant:
        print(f"    Direction   : {direction}")
    strength = (
        'negligible' if abs(spearman_r) < 0.1 else
        'weak'       if abs(spearman_r) < 0.3 else
        'moderate'   if abs(spearman_r) < 0.5 else
        'strong'
    )
    print(f"    Strength    : {strength} (|r| = {abs(spearman_r):.4f})")

    return {
        'label'      : label,
        'n'          : len(df),
        'pearson_r'  : pearson_r,
        'pearson_p'  : pearson_p,
        'spearman_r' : spearman_r,
        'spearman_p' : spearman_p,
        'slope'      : slope,
        'intercept'  : intercept,
        'r_squared'  : r_value ** 2
    }

results_men   = compute_correlations(men_df,   'ATP MEN')
results_women = compute_correlations(women_df, 'WTA WOMEN')





Combined table size: 49,813 rows
Rank distribution (raw):
count    49215.000000
mean       655.443686
std        441.312418
min          1.000000
25%        279.000000
50%        579.000000
75%       1014.000000
90%       1322.000000
95%       1430.000000
99%       1496.000000
max       1858.000000
Name: current_rank, dtype: float64

  Height column — median value: 1.83
  Height column — min: 1.57, max: 2.08
  → Heights appear to be in METRES. Converting to centimetres.

  Unique players after deduplication: 2,638

  ATP men with valid height + rank   : 1,085
  WTA women with valid height + rank : 253

───────────────────────────────────────────────────────
  CORRELATION RESULTS — ATP MEN
───────────────────────────────────────────────────────
  Sample size  : 1,085 players
  Height range : 163cm – 208cm
  Rank range   : 1 – 1858

  Pearson r    : -0.1150  (p = 0.0001)
  Spearman r   : -0.1228  (p = 0.0000)

  Primary result (Spearman):
    Correlation : -0.1228
    Significant : YES ✓

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# HEIGHT BAND ANALYSIS
# Bin players into height brackets and compute average rank per bracket
# This reveals if the relationship is non-linear
# e.g. maybe 185–195cm is the sweet spot, with both shorter
# AND taller players ranking lower
# ─────────────────────────────────────────────────────────────────────────────
def height_band_analysis(df, bin_size=5):
    df = df.copy()
    h_min = int(df['height'].min() // bin_size * bin_size)
    h_max = int(df['height'].max() // bin_size * bin_size) + bin_size
    bins  = range(h_min, h_max + bin_size, bin_size)

    df['height_band'] = pd.cut(df['height'], bins=bins)
    band_stats = (
        df.groupby('height_band', observed=True)
        .agg(
            avg_rank   = ('current_rank', 'mean'),
            median_rank= ('current_rank', 'median'),
            count      = ('player_id',    'count'),
            avg_height = ('height',       'mean')
        )
        .reset_index()
        .dropna(subset=['avg_rank'])
    )
    # Only keep bands with at least 3 players for statistical reliability
    band_stats = band_stats[band_stats['count'] >= 3]
    return band_stats

men_bands   = height_band_analysis(men_df)
women_bands = height_band_analysis(women_df)

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# PRINT CLEANING REPORT
# ─────────────────────────────────────────────────────────────────────────────
total_clean   = len(player_df)

print("\n\n" + "=" * 65)
print("DATA CLEANING REPORT — Height vs Ranking Correlation")
print("=" * 65)
print(f"  Combined rows (home + away)   : {total_original:,}")
print(f"  Unique players after cleaning : {total_clean:,}")
print(f"    ATP men                     : {len(men_df):,}")
print(f"    WTA women                   : {len(women_df):,}")
print("=" * 65)
print(pd.DataFrame(report).to_string(index=False))



DATA CLEANING REPORT — Height vs Ranking Correlation
  Combined rows (home + away)   : 49,813
  Unique players after cleaning : 1,338
    ATP men                     : 1,085
    WTA women                   : 253
 Step                                                 Description                   Rows Removed % of Original                                           Note
    2                                 Null player_id rows dropped                              0           0.0                                               
    3                Heights converted from metres to centimetres                              0           0.0              Detected metres: median was 1.83m
    4                Rows with null or unrecognised gender labels                             68          0.14                                               
    5       Invalid height values nulled (out of realistic range)                24 cells nulled          0.05              Men: 160–215cm | Women: 155–20